In [5]:
SEND_EMAIL     = True          # False -> opens a draft instead of sending
EMAIL_BOTH     = True

In [2]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import backtrader as bt
from openpyxl import load_workbook
import tempfile, os
from statsmodels.regression.rolling import RollingOLS
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import random
import pytz
import time
import os
from xbbg import blp
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown, HBox, VBox, Button, Output, Text, widgets
import sympy as sp
from sklearn.metrics import r2_score
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from IPython import get_ipython
import matplotlib.dates as mdates
from pydataquery import DataQuery
import re
import statsmodels.api as sm
from scipy.optimize import minimize
import scipy.stats as stats
import itertools
import warnings
from statsmodels.tsa.seasonal import seasonal_decompose
import yfinance as yf
import csv
import uuid
from concurrent.futures import ThreadPoolExecutor
import warnings
from multiprocess import Pool
import time
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_excel("Z1.xlsx")[['Security','Description','Asset Class', 'Geography','Instrument Type','Asset Class Tag']]
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['Description'].apply(lambda x: x.split(' ')[-1]=='Z')].sort_values(by='Description').set_index('Security')
df = df.sort_values(by=['Asset Class', 'Geography','Instrument Type','Asset Class Tag'])

pd.set_option('display.max_rows',None)
pd.reset_option('display.max_rows')

x1 = blp.bdh(tickers=list(df.index), flds='px_last', start_date='2010-1-1')

x = x1.copy()
x.columns = [item[0] for item in x.columns]   # flatten (ticker, field) -> ticker
x.index = pd.to_datetime(x.index)

# latest z-score per security (last non-NaN observation)
zsc = x.ffill().iloc[-1].round(1).rename('ZSc')

# half-life of mean reversion (in days) per security via OU/AR(1) fit
def half_life(series, dt=1):
    s = series.dropna()
    if len(s) < 3:
        return np.nan
    y = s.values[1:]
    x_lag = sm.add_constant(s.values[:-1])
    res = sm.OLS(y, x_lag).fit()
    alpha, beta = res.params
    if beta <= 0 or beta >= 1:        # no/insufficient mean reversion
        return np.nan
    kappa = -np.log(beta) / dt
    return np.log(2) / kappa

hl = x.apply(half_life).round(1).rename('HalfLife')

# rich -> cheap : highest ZSc at top, most negative at bottom
df = pd.concat([df, zsc, hl], axis=1).sort_values(by='ZSc', ascending=False)

In [ ]:
import os
import tempfile
import pandas as pd
import numpy as np
import dataframe_image as dfi
import win32com.client

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
ROWS_PER_TABLE = 50

TO_ADDR        = 'spyros.michas@iiicm.com'
CC_ADDR        = 'vasu.sharma@iiicm.com'              # "me"
SUBJECT        = f'MarFA Z-Score Tables ({pd.Timestamp.now():%d-%b-%Y}) (auto)'
ZSC_COL        = 'ZSc'
HL_COL         = 'HalfLife'
ZSC_FILTER     = None          # e.g. 2.0 to keep only |ZSc| >= 2 ; None = all rows

# ---------------------------------------------------------------
# 1) PREP
# ---------------------------------------------------------------
work = df.copy()
work[ZSC_COL] = pd.to_numeric(work[ZSC_COL], errors='coerce')
work = work[~work[ZSC_COL].isna()].copy()
if ZSC_FILTER is not None:
    work = work[work[ZSC_COL].abs() >= ZSC_FILTER].copy()
if work.index.name is None:
    work.index.name = 'Ticker'

# --- remove "BNPP MarFA" from Description so it's more legible ---
work['Description'] = (work['Description']
                       .astype(str)
                       .str.replace(r'\bBNPP\s+MarFA\b', '', regex=True)
                       .str.replace(r'\s{2,}', ' ', regex=True)
                       .str.strip())

# ---------------------------------------------------------------
# 2) GLOBAL COLOR SCALE (green = most negative, red = most positive)
# ---------------------------------------------------------------
vmin, vmax = float(work[ZSC_COL].min()), float(work[ZSC_COL].max())

def make_zsc_color(vmin, vmax):
    def _color(val):
        if not isinstance(val, (int, float, np.floating)) or pd.isna(val):
            return ''
        if val < 0 and vmin < 0:                         # negative -> green
            frac = max(min(val / vmin, 1), 0)
            r, g, b = (int(255 - (255 - 87) * frac),
                       int(255 - (255 - 187) * frac),
                       int(255 - (255 - 138) * frac))
            return f'background-color: rgba({r},{g},{b},0.75)'
        elif val > 0 and vmax > 0:                       # positive -> red
            frac = max(min(val / vmax, 1), 0)
            r, g, b = (int(255 - (255 - 230) * frac),
                       int(255 - (255 - 135) * frac),
                       int(255 - (255 - 115) * frac))
            return f'background-color: rgba({r},{g},{b},0.75)'
        return ''
    return _color

zsc_color = make_zsc_color(vmin, vmax)

def style_chunk(chunk):
    fmt = {ZSC_COL: '{:.1f}'}
    if HL_COL in chunk.columns:
        fmt[HL_COL] = lambda v: '' if pd.isna(v) else f'{v:.1f}'
    return (chunk.style
            .applymap(zsc_color, subset=[ZSC_COL])
            .applymap(lambda v: 'font-weight: bold', subset=pd.IndexSlice[:, [ZSC_COL]])
            .format(fmt)
            .set_properties(**{'text-align': 'center'})
            .set_table_styles([
                {'selector': 'tr',     'props': [('width', '8px')]},
                {'selector': 'th, td', 'props': [('line-height', '8px')]},
            ]))

# ---------------------------------------------------------------
# 3) BUILD IMAGES — one section per Asset Class, ranked rich->cheap
# ---------------------------------------------------------------
tmp_paths = []

def build_images(df_sorted):
    paths = []
    for i in range(0, len(df_sorted), ROWS_PER_TABLE):
        styled = style_chunk(df_sorted.iloc[i:i + ROWS_PER_TABLE])
        fd, path = tempfile.mkstemp(suffix='.png')
        os.close(fd)
        dfi.export(styled, path, table_conversion='chrome', dpi=100)
        paths.append(path)
        tmp_paths.append(path)
    return paths

# order asset classes by their richest (max ZSc) so hottest sits first
asset_class_order = (work.groupby('Asset Class')[ZSC_COL]
                         .max()
                         .sort_values(ascending=False)
                         .index.tolist())

sections = []
for ac in asset_class_order:
    ac_df = work[work['Asset Class'] == ac].sort_values(ZSC_COL, ascending=False)  # rich top, cheap bottom
    sections.append((ac, build_images(ac_df)))

# ---------------------------------------------------------------
# 4) ONE EMAIL, RED ASSET-CLASS HEADERS + ITS TABLES, THEN DELETE TEMPS
# ---------------------------------------------------------------
try:
    outlook = win32com.client.Dispatch("Outlook.Application")
    mail    = outlook.CreateItem(0)

    html = '<html><body>'
    for title, imgs in sections:
        html += f'<h2 style="color:#c0392b;">{title}</h2>'
        for k, path in enumerate(imgs):
            cid = f'tbl_{k}_' + os.path.splitext(os.path.basename(path))[0]
            att = mail.Attachments.Add(Source=path)
            att.PropertyAccessor.SetProperty(
                "http://schemas.microsoft.com/mapi/proptag/0x3712001F", cid)   # PR_ATTACH_CONTENT_ID
            html += f'<img src="cid:{cid}"><br><br>'
        html += '<br>'
    html += '</body></html>'

    mail.Subject  = SUBJECT
    mail.CC       = CC_ADDR
    mail.HTMLBody = html
    if EMAIL_BOTH:
        mail.To = TO_ADDR

    if SEND_EMAIL and (EMAIL_BOTH or mail.CC):
        mail.Send()
        print(f'Sent: {len(work)} rows across {len(sections)} asset classes, '
              f'{len(tmp_paths)} inline images. (EMAIL_BOTH={EMAIL_BOTH})')

finally:
    for p in tmp_paths:
        try:
            os.remove(p)
        except OSError:
            pass
    print(f'Deleted {len(tmp_paths)} temp files.')